# 第 19 课：N-gram 语言模型——概率、平滑、困惑度与回退

声学模型回答“听起来像什么”，语言模型回答“哪串 token 更自然”。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 语言模型与 WFST |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 18 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | N-gram、平滑与回退、困惑度 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：N-gram、平滑与回退、困惑度。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

from collections import Counter
from math import log,exp
from ipywidgets import interact, FloatSlider
corpus=["今天天气很好","今天天气不错","明天天气很好","今天心情很好","北京天气很好","北京天气不错"]

项目根目录: G:\learn_asr


## 1. Bigram：下一个字符只看前一个字符

In [2]:
uni=Counter();bi=Counter();vocab=set()
for s in corpus:
    seq=["<s>"]+list(s)+["</s>"];vocab.update(seq[1:])
    for a,b in zip(seq,seq[1:]): uni[a]+=1;bi[a,b]+=1
V=len(vocab)
def p_bigram(b,a,k=.1): return (bi[a,b]+k)/(uni[a]+k*V)
for b in ["天","心","北","好"]: print(f"P({b}|今)={p_bigram(b,'今'):.4f}")

P(天|今)=0.7209
P(心|今)=0.0233
P(北|今)=0.0233
P(好|今)=0.0233


## 2. 句子概率要在 log-space 相加

In [3]:
def sentence_logp(s,k=.1):
    seq=["<s>"]+list(s)+["</s>"]
    return sum(log(p_bigram(b,a,k)) for a,b in zip(seq,seq[1:]))
for s in ["今天天气很好","今天心情不错","北京心情很好"]: print(s,sentence_logp(s))

今天天气很好 -4.309917502723476
今天心情不错 -8.197582269848969
北京心情很好 -7.183067268591691


## 3. 困惑度 Perplexity

$$PPL=\exp\left(-\frac{1}{N}\sum_i\log P(w_i|history)\right)$$

越低表示模型对测试文本越不“意外”，但不同 tokenization、词表和测试集的 PPL 不宜直接比较。

In [4]:
def ppl(s): return exp(-sentence_logp(s)/(len(s)+1))
for s in ["今天天气很好","今天心情不错","北京心情很好"]: print(s,ppl(s))

今天天气很好 1.850956440555105
今天心情不错 3.2254845318835947
北京心情很好 2.79030934754096


## 4. 平滑为什么必要

In [5]:
for k in [0,1e-3,.1,1.0]:
    try: print("k",k,"P(北|今)",p_bigram("北","今",k))
    except ZeroDivisionError: print("division error")

k 0 P(北|今) 0.0
k 0.001 P(北|今) 0.00033189512114171923
k 0.1 P(北|今) 0.023255813953488375
k 1.0 P(北|今) 0.0625


真实系统常用 modified Kneser–Ney、backoff 与 ARPA 格式。本课的 add-k 只是看懂概率用的教学版本。

## 5. LM scale 与 acoustic score

In [6]:
candidates={"今天天气很好":-9.8,"今天心情很好":-9.2}
@interact(alpha=FloatSlider(min=0,max=2,value=.5,step=.1,description="LM scale"))
def combine(alpha=.5):
    for s,am in candidates.items(): print(s,"AM",am,"LM",sentence_logp(s),"total",am+alpha*sentence_logp(s))

interactive(children=(FloatSlider(value=0.5, description='LM scale', max=2.0), Output()), _dom_classes=('widge…

## 本课测试

1. 语言模型是否直接听声音？
2. 未见过的 bigram 为什么需要平滑？
3. PPL 越低是否总能保证 WER 越低？
4. LM scale 太大有什么风险？
5. 字级 LM 和词级 LM 的词典需求有何不同？

<details><summary>展开参考答案</summary>

1. 不，它对 token 序列打分。2. 避免概率为零。3. 不保证，解码组合、领域和声学候选都会影响 WER。4. 模型可能忽视声音而偏向常见句。5. 词级通常依赖分词与 OOV 处理；字级词表更简单但序列更长。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 19 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `N-gram`、`平滑与回退`、`困惑度`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**测试句含未见 bigram**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**实现 add-k bigram 并计算句子 log probability**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**说明 LM 分数怎样进入 beam/WFST**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：N-gram、平滑与回退、困惑度。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 N-gram、平滑与回退、困惑度。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
